In [118]:
# import needed packages
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm


In [119]:
# set the device

if torch.backends.mps.is_available() :
    device = torch.device("mps")
    print(f"Training on device : {device}")
else :
    print("Apple Silicon GPU not available. Training on CPU.")

Training on device : mps


In [120]:
# fetch the dataset and build the loader

data_path = "./data"

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees = 15,
        translate = (0.1, 0.1),
        scale = (0.8, 1.2)
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root = data_path,
    train = True,
    download = True,
    transform = train_transform
)
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)

val_dataset = torchvision.datasets.MNIST(
    root = data_path,
    train = False,
    download = True,
    transform = val_transform
)
val_loader = DataLoader(val_dataset, batch_size = 1000, shuffle = False)

In [122]:
# define the model, loss function, and optimizer
from models.basic_classifier import BasicClassifier

model = BasicClassifier(num_classes = 10)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [124]:
# define the training loop
from pathlib import Path
from utilities.train import train_step, evaluate

num_epochs = 5
best_model = None
best_accuracy = 0

for epoch in range(num_epochs) :

    # instantiate training progress bar
    pbar = tqdm(train_loader, desc = f"Epoch {epoch + 1}/{num_epochs}", leave = True)

    # train 1 epoch on training set
    model, avg_epoch_loss, train_accuracy = train_step(model, loss_function, optimizer,device, pbar)

    # evaluate on val test
    val_accuracy = evaluate(model, val_loader, device)
    print(f"Training Accuracy: {train_accuracy * 100 :.2f} % Validation Accuracy: {val_accuracy * 100:.2f} %")

    if val_accuracy > best_accuracy :
        best_model = model
        best_acccuracy = val_accuracy

# save the best model
save_dir= Path("model_params")
best_model_path = save_dir / "my_best_model.pt"
torch.save(best_model.state_dict(), best_model_path)

Epoch 1/5: 100%|██████████| 938/938 [00:06<00:00, 153.91it/s, loss=0.53]  


Training Accuracy: 92.37 % Validation Accuracy: 96.66 %


Epoch 2/5: 100%|██████████| 938/938 [00:06<00:00, 155.41it/s, loss=0.217] 


Training Accuracy: 93.17 % Validation Accuracy: 97.27 %


Epoch 3/5: 100%|██████████| 938/938 [00:06<00:00, 152.24it/s, loss=0.37]  


Training Accuracy: 93.63 % Validation Accuracy: 97.30 %


Epoch 4/5: 100%|██████████| 938/938 [00:06<00:00, 154.62it/s, loss=0.0436]


Training Accuracy: 94.06 % Validation Accuracy: 97.59 %


Epoch 5/5: 100%|██████████| 938/938 [00:06<00:00, 153.45it/s, loss=0.177] 


Training Accuracy: 94.27 % Validation Accuracy: 97.68 %
